In [3]:
!pip install -q ultralytics
import ultralytics
import os

print(os.path.dirname(ultralytics.__file__))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
/usr/local/lib/python3.12/dist-packages/ultralytics


In [ ]:
import ultralytics
import os

print(os.path.dirname(ultralytics.__file__))

In [7]:
import os
import ultralytics

root = os.path.dirname(ultralytics.__file__)

models_dir = os.path.join(root, "cfg", "models")

for path, dirs, files in os.walk(models_dir):
    print(path)
    for f in files:
        print("   ", f)

/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/models
/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/models/v10
    yolov10s.yaml
    yolov10b.yaml
    yolov10n.yaml
    yolov10x.yaml
    yolov10m.yaml
    yolov10l.yaml
/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/models/11
    yoloe-11.yaml
    yoloe-11-seg.yaml
    yolo11-pose.yaml
    yolo11-cls-resnet18.yaml
    yolo11-seg.yaml
    yolo11-cls.yaml
    yolo11-obb.yaml
    yolo11.yaml
/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/models/12
    yolo12.yaml
    yolo12-obb.yaml
    yolo12-pose.yaml
    yolo12-seg.yaml
    yolo12-cls.yaml
/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/models/v3
    yolov3-tiny.yaml
    yolov3-spp.yaml
    yolov3.yaml
/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/models/rt-detr
    rtdetr-x.yaml
    rtdetr-l.yaml
    rtdetr-resnet101.yaml
    rtdetr-resnet50.yaml
/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/models/v8
    yolov8-pose.yaml

In [8]:
import os
import ultralytics

root = os.path.dirname(ultralytics.__file__)

yaml_path = os.path.join(root, "cfg/models/11/yolo11.yaml")

print(yaml_path)
with open(yaml_path, "r") as f:
    print(f.read())

/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/models/11/yolo11.yaml
# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

# Ultralytics YOLO11 object detection model with P3/8 - P5/32 outputs
# Model docs: https://docs.ultralytics.com/models/yolo11
# Task docs: https://docs.ultralytics.com/tasks/detect

# Parameters
nc: 80 # number of classes
scales: # model compound scaling constants, i.e. 'model=yolo11n.yaml' will call yolo11.yaml with scale 'n'
  # [depth, width, max_channels]
  n: [0.50, 0.25, 1024] # summary: 181 layers, 2624080 parameters, 2624064 gradients, 6.6 GFLOPs
  s: [0.50, 0.50, 1024] # summary: 181 layers, 9458752 parameters, 9458736 gradients, 21.7 GFLOPs
  m: [0.50, 1.00, 512] # summary: 231 layers, 20114688 parameters, 20114672 gradients, 68.5 GFLOPs
  l: [1.00, 1.00, 512] # summary: 357 layers, 25372160 parameters, 25372144 gradients, 87.6 GFLOPs
  x: [1.00, 1.50, 512] # summary: 357 layers, 56966176 parameters, 56966160 gradients, 196.0 GFLOP

In [1]:
%%writefile yolo11_p2.yaml
# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

# YOLO11 + ATL (Tiny Object Detection Layer / P2 head)
# Backbone: IDENTICAL to official YOLO11
# Neck: FPN extended to P2, PAN extended back through P2→P3→P4→P5
# Detect: P2 / P3 / P4 / P5  (4-head)

nc: 80
scales:
  # [depth, width, max_channels]
  n: [0.50, 0.25, 1024]
  s: [0.50, 0.50, 1024]
  m: [0.50, 1.00,  512]
  l: [1.00, 1.00,  512]
  x: [1.00, 1.50,  512]

# ── Backbone (UNCHANGED) ────────────────────────────────────────────────────
backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv,   [64, 3, 2]]           # 0  P1/2
  - [-1, 1, Conv,   [128, 3, 2]]          # 1  P2/4
  - [-1, 2, C3k2,   [256, False, 0.25]]   # 2  P2/4  ← tiny-object feature
  - [-1, 1, Conv,   [256, 3, 2]]          # 3  P3/8
  - [-1, 2, C3k2,   [512, False, 0.25]]   # 4  P3/8  ← small feature
  - [-1, 1, Conv,   [512, 3, 2]]          # 5  P4/16
  - [-1, 2, C3k2,   [512, True]]          # 6  P4/16 ← medium feature
  - [-1, 1, Conv,   [1024, 3, 2]]         # 7  P5/32
  - [-1, 2, C3k2,   [1024, True]]         # 8  P5/32
  - [-1, 1, SPPF,   [1024, 5]]            # 9
  - [-1, 2, C2PSA,  [1024]]               # 10 P5/32 ← large feature

# ── Head ────────────────────────────────────────────────────────────────────
head:
  # ── FPN top-down: P5 → P4 ──────────────────────────────────────────────
  - [-1,       1, nn.Upsample, [None, 2, "nearest"]]  # 11  P5→P4 res
  - [[-1, 6],  1, Concat,      [1]]                   # 12  cat backbone P4
  - [-1,       2, C3k2,        [512, False]]           # 13  fused P4

  # ── FPN top-down: P4 → P3 ──────────────────────────────────────────────
  - [-1,       1, nn.Upsample, [None, 2, "nearest"]]  # 14  P4→P3 res
  - [[-1, 4],  1, Concat,      [1]]                   # 15  cat backbone P3
  - [-1,       2, C3k2,        [256, False]]           # 16  fused P3

  # ── FPN top-down: P3 → P2  (NEW) ───────────────────────────────────────
  - [-1,       1, nn.Upsample, [None, 2, "nearest"]]  # 17  P3→P2 res
  - [[-1, 2],  1, Concat,      [1]]                   # 18  cat backbone P2 (layer 2)
  - [-1,       2, C3k2,        [128, False]]           # 19  fused P2  ← P2/4-tiny  (NEW)

  # ── PAN bottom-up: P2 → P3  (NEW) ──────────────────────────────────────
  - [-1,        1, Conv,   [128, 3, 2]]               # 20  P2/4 → P3/8  (NEW)
  - [[-1, 16],  1, Concat, [1]]                       # 21  cat fused P3 (layer 16)  (NEW)
  - [-1,        2, C3k2,   [256, False]]              # 22  PAN-P3  (NEW)

  # ── PAN bottom-up: P3 → P4 ─────────────────────────────────────────────
  - [-1,        1, Conv,   [256, 3, 2]]               # 23  P3/8 → P4/16
  - [[-1, 13],  1, Concat, [1]]                       # 24  cat fused P4 (layer 13)
  - [-1,        2, C3k2,   [512, False]]              # 25  PAN-P4

  # ── PAN bottom-up: P4 → P5 ─────────────────────────────────────────────
  - [-1,        1, Conv,   [512, 3, 2]]               # 26  P4/16 → P5/32
  - [[-1, 10],  1, Concat, [1]]                       # 27  cat backbone P5 (layer 10)
  - [-1,        2, C3k2,   [1024, True]]              # 28  PAN-P5

  # ── Detect: P2 / P3 / P4 / P5 ──────────────────────────────────────────
  - [[19, 22, 25, 28], 1, Detect, [nc]]               # 29  Detect(P2,P3,P4,P5)

Writing yolo11_p2.yaml


In [3]:
!pip install -q ultralytics
from ultralytics import YOLO

model = YOLO("/kaggle/working/yolo11_p2.yaml")
model.info()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
WARNING ⚠️ no model scale passed. Assuming scale='n'.
YOLO11_p2 summary: 217 layers, 2,740,032 parameters, 2,740,016 gradients, 11.4 GFLOPs


(217, 2740032, 2740016, 11.4099328)